# 09. Graphs: traversal, shortest paths, ordering and spanning trees

**Goal.** Model a problem as a graph, pick the matching algorithm and state its cost.

**You will be able to:**
1. Store a graph as an adjacency list and explain when a matrix is better.
2. Run BFS, DFS, Dijkstra and A* and justify each choice.
3. Order tasks with a topological sort and detect cycles.
4. Build a minimum spanning tree with union-find and Kruskal.

The examples use a table of approximate UK road distances (`understanding_python.pathfinding`). Straight-line distance never exceeds road distance, which makes it a valid A* heuristic.

## 1. Representation, BFS and DFS

### How it works under the hood
- An **adjacency list** maps each vertex to its neighbours: O(V + E) memory. An **adjacency matrix** is a V by V table: O(V^2) memory, O(1) edge test.
- **BFS** uses a FIFO queue. It visits vertices in rings of equal hop count. The first visit to a vertex therefore uses a fewest-edges path. Cost O(V + E).
- **DFS** uses a LIFO stack (or recursion). It dives down one branch before backtracking. Cost O(V + E). It finds connected components, cycles and orderings.
- Recursive DFS uses one stack frame per level. CPython stops at about 1000 frames (`sys.getrecursionlimit()`), so use an explicit stack for large graphs.
- Mark vertices visited when you enqueue them, or the queue fills with duplicates.

### Production use
- BFS: fewest hops in a social network ("people you may know"), web crawlers, level-by-level dependency scans, shortest paths on unweighted maps.
- DFS: finding connected components, detecting deadlocks, solving mazes, exploring file trees.

### Trade-offs

| Representation | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Adjacency list (`dict` of lists) | O(V + E) | Iterate neighbours in O(degree) | Natural in Python | Compact on sparse graphs | Edge test is O(degree) |
| Adjacency matrix | O(V^2) | O(1) edge test | Simple indexing | Good for dense graphs | Wasteful when sparse |
| Edge list | O(E) | Scan to find neighbours | Easiest to build | Ideal for Kruskal | Slow neighbour lookup |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import sys
import time

from understanding_python import graphs as gr
from understanding_python.pathfinding import CITY_COORDS, astar, haversine_km, uk_road_graph

roads = uk_road_graph()
plain = {city: [n for n, _ in edges] for city, edges in roads.items()}
edge_count = sum(len(v) for v in roads.values()) // 2
print(f"{len(roads)} cities, {edge_count} roads (each stored in both directions)")

hops = gr.bfs(plain, "London")
by_ring = {}
for city, d in hops.items():
    by_ring.setdefault(d, []).append(city)
for d in sorted(by_ring):
    print(f"  {d} hop(s) from London: {', '.join(sorted(by_ring[d]))}")

order = gr.dfs(plain, "London")
print("DFS preorder from London:", " > ".join(order[:7]), "...")

components = []
seen = set()
for city in plain:
    if city not in seen:
        comp = gr.dfs(plain, city)
        seen.update(comp)
        components.append(comp)
print("connected components:", len(components))

n = 3000
sparse = {i: [(i + 1) % n, (i * 7 + 3) % n] for i in range(n)}
matrix_cells = n * n
print(f"{n} vertices, {sum(map(len, sparse.values()))} edges: list entries {sum(map(len, sparse.values())):,} vs matrix cells {matrix_cells:,}")

depth = 5000
chain = {i: [i + 1] for i in range(depth)}
chain[depth] = []
print("iterative DFS reaches the end of a", depth, "node chain:", gr.dfs(chain, 0)[-1], "| recursion limit is", sys.getrecursionlimit())

13 cities, 23 roads (each stored in both directions)
  0 hop(s) from London: London
  1 hop(s) from London: Birmingham, Bristol, Nottingham
  2 hop(s) from London: Cardiff, Leeds, Liverpool, Manchester, Sheffield
  3 hop(s) from London: Carlisle, Newcastle
  4 hop(s) from London: Edinburgh, Glasgow
DFS preorder from London: London > Bristol > Cardiff > Birmingham > Nottingham > Sheffield > Manchester ...
connected components: 1
3000 vertices, 6000 edges: list entries 6,000 vs matrix cells 9,000,000
iterative DFS reaches the end of a 5000 node chain: 5000 | recursion limit is 1000


## 2. Dijkstra's algorithm

### How it works under the hood
- Dijkstra finds the cheapest path from one source to every vertex when all edge weights are non-negative.
- It keeps a priority queue of `(distance, vertex)`. It pops the closest unsettled vertex, then **relaxes** each outgoing edge: if `dist[u] + w < dist[v]`, update `v` and push it.
- With a binary heap the cost is O((V + E) log V). The heap holds stale entries instead of supporting decrease-key, so the code skips vertices already settled.
- A negative edge can lower the distance of an already settled vertex, which breaks the greedy argument. Use Bellman-Ford for negative weights.
- Store a `prev` map to rebuild the path by walking backwards from the target.

### Production use
- Road and logistics routing, network packet routing (OSPF uses it), game pathfinding, cheapest-fare search.
- Any "cheapest way to reach X" problem with non-negative costs.

### Trade-offs

| Algorithm | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| BFS | O(V) queue | O(V + E) | Simplest | Optimal for unweighted graphs | Ignores weights |
| Dijkstra (heap) | O(V + E) | O((V + E) log V) | Moderate | Optimal, all targets at once | Needs non-negative weights, explores in all directions |
| Bellman-Ford | O(V) | O(V * E) | Simple | Handles negative edges, detects negative cycles | Slow |
| Floyd-Warshall | O(V^2) | O(V^3) | Very short | All pairs at once | Only for small graphs |

In [3]:
dist, prev = gr.dijkstra(roads, "London")
print("cheapest distance from London (km):")
for city, d in sorted(dist.items(), key=lambda kv: kv[1])[1:8]:
    print(f"  {city:11} {d:6.0f}")

path = gr.path_to(prev, "London", "Glasgow")
legs = [dict(roads[a])[b] for a, b in zip(path, path[1:])]
print("London to Glasgow:", " > ".join(path))
print("legs (km):", legs, "| total:", sum(legs), "| matches dist:", sum(legs) == dist["Glasgow"])
print("straight-line London to Glasgow:", round(haversine_km(CITY_COORDS["London"], CITY_COORDS["Glasgow"])), "km")

try:
    gr.dijkstra({"a": [("b", -4.0)], "b": []}, "a")
except ValueError as exc:
    print("negative weight rejected:", exc)

island = dict(roads, Lerwick=[])
d2, _ = gr.dijkstra(island, "London")
print("unreachable city missing from result:", "Lerwick" not in d2)

cheapest distance from London (km):
  Bristol        190
  Birmingham     190
  Nottingham     200
  Sheffield      255
  Cardiff        260
  Leeds          310
  Manchester     315
London to Glasgow: London > Nottingham > Sheffield > Manchester > Carlisle > Glasgow
legs (km): [200, 55, 60, 190, 155] | total: 660 | matches dist: True
straight-line London to Glasgow: 555 km
negative weight rejected: negative edge weight
unreachable city missing from result: True


## 3. A* search

### How it works under the hood
- A* is Dijkstra with a compass. It orders the queue by `g(n) + h(n)`: the cost so far plus a **heuristic** estimate of the cost still to go.
- If `h` never overestimates (it is **admissible**), A* returns an optimal path. If `h` is also **consistent**, each vertex settles once.
- With `h = 0` A* is exactly Dijkstra. A better `h` prunes more of the search space.
- Straight-line distance suits maps. Manhattan distance suits 4-direction grids. An overestimating `h` runs faster but can return a worse path.
- The worst case is still exponential in the path length for bad heuristics. Memory is the usual limit: A* keeps the whole frontier.

### Production use
- Game and robot navigation, map routing with landmarks, puzzle solvers, route planners that need one target quickly.

### Trade-offs

| Method | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| A* with admissible `h` | Frontier plus closed set | Fewer expansions than Dijkstra | Dijkstra plus one function | Optimal and focused | Needs a good heuristic |
| Dijkstra | Same | Explores a disc around the source | Simplest | No heuristic needed | Wasted work for single targets |
| Greedy best-first | Smaller | Fastest | Simple | Very fast | Not optimal |
| Weighted A* (`h` scaled up) | Same | Fewer expansions | Easy tweak | Speed on huge maps | Bounded suboptimal paths |

In [4]:
import random

from understanding_python.pathfinding import grid_neighbours, manhattan

goal = "Glasgow"
h_road = lambda c: haversine_km(CITY_COORDS[c], CITY_COORDS[goal])
c_a, p_a, e_a = astar(lambda c: roads[c], "London", goal, h_road)
c_d, p_d, e_d = astar(lambda c: roads[c], "London", goal, lambda c: 0)
print(f"road network : A* {c_a:.0f} km, {e_a} expansions | Dijkstra {c_d:.0f} km, {e_d} expansions | same path: {p_a == p_d}")

rng = random.Random(8)
SIZE = 41
grid = [["#" if rng.random() < 0.22 else "." for _ in range(SIZE)] for _ in range(SIZE)]
start, end = (0, 0), (SIZE - 1, SIZE - 1)
for r, c in (start, end):
    grid[r][c] = "."
rows = ["".join(r) for r in grid]
nb = grid_neighbours(rows)

cost_a, path_a, exp_a = astar(nb, start, end, manhattan(end))
cost_d, path_d, exp_d = astar(nb, start, end, lambda p: 0)
print(f"grid {SIZE}x{SIZE}  : A* cost {cost_a:.0f}, {exp_a} expansions | Dijkstra cost {cost_d:.0f}, {exp_d} expansions | saving {100 * (1 - exp_a / exp_d):.0f}%")

cost_w, path_w, exp_w = astar(nb, start, end, lambda p: 3 * manhattan(end)(p))
print(f"overestimating h (x3): cost {cost_w:.0f} vs optimal {cost_a:.0f}, {exp_w} expansions")

on_path = set(path_a)
print("first 12 rows of the grid, # = wall, * = A* path:")
for r in range(0, 12):
    print("".join("*" if (r, c) in on_path else rows[r][c] for c in range(SIZE)))

print("sealed target:", astar(grid_neighbours(["..#.", "..#.", "..#."]), (0, 0), (0, 3), manhattan((0, 3)))[0])

road network : A* 660 km, 10 expansions | Dijkstra 660 km, 13 expansions | same path: True
grid 41x41  : A* cost 80, 870 expansions | Dijkstra cost 80, 1306 expansions | saving 33%
overestimating h (x3): cost 96 vs optimal 80, 133 expansions
first 12 rows of the grid, # = wall, * = A* path:
*.#.#..#....#.#.#....#..#.........##..#..
**...#.#.....##..#...#.#......#.#..#..#..
#**.....###......###.....#.#.#....#..#...
##*.#.......#.#...#..#....#....#.........
..*......#..................#...#........
..*..............#..#....#....#..........
..*#..#....#..#.##............#..#.#.....
..*.........#......#....#.#.....#.##.#...
.#*#.......#.......##...#................
..*.........##.#..#...###.#......#.....#.
#.*.....##........###.........#..#.......
.#**...........#....#.....#......#.......
sealed target: inf


## 4. Topological sort

### How it works under the hood
- A topological order lists every vertex after all vertices it depends on. It exists only for a **directed acyclic graph** (DAG).
- **Kahn's algorithm**: count each vertex's in-degree, queue those with zero, then repeatedly pop one, emit it and decrement its neighbours. O(V + E).
- If fewer than V vertices were emitted, a cycle blocks the rest. The sort doubles as a cycle detector.
- A DFS-based version emits vertices in reverse finishing order. Both give valid orders. Many valid orders can exist.

### Production use
- Build systems (Make, Bazel), package installers resolving dependencies, task schedulers (Airflow DAGs), spreadsheet recalculation, course prerequisites.

### Trade-offs

| Method | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Kahn (in-degree queue) | O(V + E) | O(V + E) | Clear, iterative | Detects cycles directly, gives parallel levels | Needs in-degree map |
| DFS finishing order | O(V) stack | O(V + E) | Short with recursion | Compact | Recursion depth, cycle check needs colours |
| Repeated scanning | O(1) extra | O(V * E) | Obvious | No structure needed | Slow |

In [5]:
build = {
    "app.exe": [],
    "ui.o": ["app.exe"],
    "net.o": ["app.exe"],
    "core.o": ["ui.o", "net.o"],
    "config.h": ["core.o", "ui.o"],
    "proto.h": ["net.o"],
}
order = gr.topological_sort(build)
print("build order:", " > ".join(order))
pos = {t: i for i, t in enumerate(order)}
print("every dependency precedes its dependant:", all(pos[a] < pos[b] for a, deps in build.items() for b in deps))

cyclic = dict(build, **{"app.exe": ["config.h"]})
try:
    gr.topological_sort(cyclic)
except ValueError as exc:
    print("cycle detected:", exc)

print("empty graph:", gr.topological_sort({}), "| single node:", gr.topological_sort({"x": []}))

build order: config.h > proto.h > core.o > ui.o > net.o > app.exe
every dependency precedes its dependant: True
cycle detected: graph has a cycle
empty graph: [] | single node: ['x']


## 5. Union-find and minimum spanning trees

### How it works under the hood
- A **minimum spanning tree** (MST) connects all vertices with the lowest total edge weight and no cycles.
- **Kruskal's algorithm** sorts edges by weight and adds an edge when its ends are in different components. Sorting costs O(E log E).
- **Union-find** (disjoint sets) answers "same component?" in near-constant time. It stores a parent pointer per element. `find` follows pointers to the root and **compresses the path**. `union` attaches the smaller tree under the larger one.
- With both optimisations each operation costs O(alpha(n)), the inverse Ackermann function, below 5 for any practical n.

### Production use
- Network design: cheapest cabling or pipeline links.
- Clustering (single-linkage), image segmentation and merging accounts that share an identifier.
- Detecting cycles while adding edges.

### Trade-offs

| Algorithm | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Kruskal + union-find | O(E) edge list plus O(V) | O(E log E) | Short | Great for sparse graphs | Needs all edges up front |
| Prim (heap) | O(V + E) | O(E log V) | Moderate | Grows from one vertex, good for dense graphs | Needs adjacency lists |
| Union-find alone | O(V) | Near O(1) per op | Tiny | Online connectivity | Cannot split sets |

In [6]:
from understanding_python.structures import UnionFind

names = sorted(roads)
index = {c: i for i, c in enumerate(names)}
edges = sorted({(min(index[a], index[b]), max(index[a], index[b]), w) for a, es in roads.items() for b, w in es})
total, used = gr.kruskal(len(names), edges)
print(f"{len(edges)} roads, cheapest network joining all {len(names)} cities uses {len(used)} of them, {total:.0f} km")
for u, v, w in sorted(used, key=lambda e: e[2]):
    print(f"  {names[u]:11} - {names[v]:11} {w:4.0f} km")
print("spanning tree has V - 1 edges:", len(used) == len(names) - 1)

uf = UnionFind(len(names))
for u, v, _ in used:
    uf.union(u, v)
print("one component after Kruskal:", uf.components == 1)
print("adding any further road closes a cycle:", all(not uf.union(u, v) for u, v, _ in edges))

23 roads, cheapest network joining all 13 cities uses 12 of them, 1175 km
  Leeds       - Sheffield     55 km
  Liverpool   - Manchester    55 km
  Nottingham  - Sheffield     55 km
  Manchester  - Sheffield     60 km
  Bristol     - Cardiff       70 km
  Edinburgh   - Glasgow       75 km
  Birmingham  - Nottingham    80 km
  Carlisle    - Newcastle     90 km
  Birmingham  - Bristol      140 km
  Leeds       - Newcastle    150 km
  Carlisle    - Edinburgh    155 km
  Birmingham  - London       190 km
spanning tree has V - 1 edges: True
one component after Kruskal: True
adding any further road closes a cycle: True
